In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!unzip -q -o "/content/drive/MyDrive/super-ai-engineer-season-6-individual-hackathon-house-recognition.zip" -d "/content/dataset/"

In [ ]:
import os
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
from pathlib import Path
# ==========================================
# 1. ระบบค้นหา Path แบบเจาะลึก (Deep Search)
# ==========================================
BASE_DIR = Path('/content/dataset')

TRAIN_IMG_DIR = BASE_DIR / "train"
TEST_IMG_DIR = BASE_DIR / "test"
TRAIN_CSV_PATH = BASE_DIR / "train.csv"
SAMPLE_SUB_PATH = BASE_DIR / "sample_submission.csv"

BATCH_SIZE = 32
EPOCHS = 15
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# ==========================================
# 2. Custom Dataset Class
# ==========================================
class HouseDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        if self.is_test:
            img_id = str(self.df.iloc[idx]['id'])
            img_name = f"{img_id}.jpg"
        else:
            img_name = str(self.df.iloc[idx]['image_name'])

        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        if self.is_test:
            return image, img_id
        else:
            label = int(self.df.iloc[idx]['class'])
            return image, torch.tensor(label, dtype=torch.long)

In [ ]:
# ==========================================
# 3. กรองไฟล์และแบ่งข้อมูล (Fixed KeyError)
# ==========================================
mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

val_test_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

# โหลดและกรองรูปภาพที่มีอยู่จริง
raw_df = pd.read_csv(TRAIN_CSV_PATH)
print("🔍 กำลังสแกนความถูกต้องของไฟล์รูปภาพ...")

# ใช้ List Comprehension เพื่อความเร็วในการเช็คไฟล์
existing_mask = [os.path.exists(os.path.join(TRAIN_IMG_DIR, str(name))) for name in raw_df['image_name']]
full_train_df = raw_df[existing_mask].copy()

if len(full_train_df) == 0:
    raise ValueError("❌ ไม่พบรูปภาพที่ตรงกับชื่อใน CSV เลย! โปรดเช็ค Path โฟลเดอร์ Train อีกครั้ง")

print(f"✅ พร้อมเทรน: {len(full_train_df)} รูป | ตกหล่น: {len(raw_df) - len(full_train_df)} รูป")

# แบ่งข้อมูล
train_df, val_df = train_test_split(full_train_df, test_size=0.2, random_state=42, stratify=full_train_df['class'])
test_df = pd.read_csv(SAMPLE_SUB_PATH)

train_loader = DataLoader(HouseDataset(train_df, TRAIN_IMG_DIR, train_transform), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(HouseDataset(val_df, TRAIN_IMG_DIR, val_test_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(HouseDataset(test_df, TEST_IMG_DIR, val_test_transform, is_test=True), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)


In [ ]:
# ==========================================
# 4. Model Setup (EfficientNetV2-S)
# ==========================================
model = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.DEFAULT)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [ ]:
# ==========================================
# 5. Training Loop
# ==========================================
best_val_acc = 0.0
print(f"\n🚀 เริ่มเทรนบน {DEVICE}...")

for epoch in range(EPOCHS):
    model.train()
    t_loss, t_corr, t_total = 0, 0, 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for inputs, labels in pbar:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        t_loss += loss.item()
        _, pred = torch.max(outputs, 1)
        t_total += labels.size(0)
        t_corr += (pred == labels).sum().item()
        pbar.set_postfix({'loss': f'{t_loss/len(train_loader):.4f}', 'acc': f'{t_corr/t_total:.4f}'})

    scheduler.step()

    # Validation
    model.eval()
    v_corr, v_total = 0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            outputs = model(inputs)
            _, pred = torch.max(outputs, 1)
            v_total += labels.size(0)
            v_corr += (pred == labels).sum().item()

    v_acc = v_corr / v_total
    print(f"✅ Epoch {epoch+1} Val Acc: {v_acc:.4f} | Best: {best_val_acc:.4f}")

    if v_acc > best_val_acc:
        best_val_acc = v_acc
        torch.save(model.state_dict(), 'best_model.pth')

# ==========================================
# 6. Inference & Submission
# ==========================================
import torch.nn.functional as F

print("\n🎯 กำลังทำนายผล Test Set ด้วยเทคนิค TTA (Test-Time Augmentation)...")
model.load_state_dict(torch.load('best_model.pth'))
model.eval()
all_preds, all_ids = [], []

with torch.no_grad():
    for inputs, ids in tqdm(test_loader):
        inputs = inputs.to(DEVICE)

        # 1. ทำนายจากรูปต้นฉบับ
        outputs1 = model(inputs)
        prob1 = F.softmax(outputs1, dim=1)

        # 2. ทำนายจากรูปแบบกลับซ้ายขวา (Flipped)
        inputs_flipped = torch.flip(inputs, dims=[3])
        outputs2 = model(inputs_flipped)
        prob2 = F.softmax(outputs2, dim=1)

        # 3. เอาความมั่นใจมาเฉลี่ยกัน (ช่วยลด Error ได้มหาศาลใน Private Board)
        avg_prob = (prob1 + prob2) / 2.0

        _, pred = torch.max(avg_prob, 1)
        all_preds.extend(pred.cpu().numpy())
        all_ids.extend(ids)

pd.DataFrame({'id': all_ids, 'answer': all_preds}).to_csv('submission_TTA.csv', index=False)
print("🎉 เสร็จเรียบร้อย! ไฟล์ submission_TTA.csv พร้อมส่งแล้วครับ")

🔍 กำลังสแกนความถูกต้องของไฟล์รูปภาพ...
✅ พร้อมเทรน: 2953 รูป | ตกหล่น: 0 รูป
Downloading: "https://download.pytorch.org/models/efficientnet_v2_s-dd5fe13b.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_s-dd5fe13b.pth


100%|██████████| 82.7M/82.7M [00:00<00:00, 138MB/s]



🚀 เริ่มเทรนบน cuda...


Epoch 1/15:   0%|          | 0/74 [00:00<?, ?it/s]

✅ Epoch 1 Val Acc: 0.9391 | Best: 0.0000


...

✅ Epoch 14 Val Acc: 0.9645 | Best: 0.9645


Epoch 15/15:   0%|          | 0/74 [00:00<?, ?it/s]

✅ Epoch 15 Val Acc: 0.9577 | Best: 0.9645

🎯 กำลังทำนายผล Test Set ด้วยเทคนิค TTA (Test-Time Augmentation)...


  0%|          | 0/49 [00:00<?, ?it/s]

🎉 เสร็จเรียบร้อย! ไฟล์ submission_TTA.csv พร้อมส่งแล้วครับ
